In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark = SparkSession.builder\
.appName("githubQuestions")\
.master("spark://172.18.0.2:7077")\
.getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/25 18:36:55 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/25 18:36:59 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [2]:
df = spark.read.format("csv").load("/opt/spark-data/menu.csv")

26/06/25 18:42:20 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
26/06/25 18:42:35 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
26/06/25 18:42:50 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
26/06/25 18:43:05 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
                                                                                

In [7]:
chipo = (spark.read
            .option("header", "true")
            .option("delimiter","\t")
            .option("truncate", "false")
            .option("inferSchema", "true")
            .csv("/opt/spark-data/menu.csv"))
            

chipo.show()

+--------+--------+--------------------+--------------------+----------+
|order_id|quantity|           item_name|  choice_description|item_price|
+--------+--------+--------------------+--------------------+----------+
|       1|       1|Chips and Fresh T...|                NULL|    $2.39 |
|       1|       1|                Izze|        [Clementine]|    $3.39 |
|       1|       1|    Nantucket Nectar|             [Apple]|    $3.39 |
|       1|       1|Chips and Tomatil...|                NULL|    $2.39 |
|       2|       2|        Chicken Bowl|[Tomatillo-Red Ch...|   $16.98 |
|       3|       1|        Chicken Bowl|[Fresh Tomato Sal...|   $10.98 |
|       3|       1|       Side of Chips|                NULL|    $1.69 |
|       4|       1|       Steak Burrito|[Tomatillo Red Ch...|   $11.75 |
|       4|       1|    Steak Soft Tacos|[Tomatillo Green ...|    $9.25 |
|       5|       1|       Steak Burrito|[Fresh Tomato Sal...|    $9.25 |
|       5|       1| Chips and Guacamole|           

In [8]:
chipo.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- item_name: string (nullable = true)
 |-- choice_description: string (nullable = true)
 |-- item_price: string (nullable = true)



In [15]:
# select item_name, sum(quantity)
# from chipo
# group by 1;

most_ordered = chipo.groupBy(col("item_name")).agg(sum(col("quantity")).alias("total_qty")).orderBy(col("total_qty").desc())

In [16]:
most_ordered.show(10)

+--------------------+---------+
|           item_name|total_qty|
+--------------------+---------+
|        Chicken Bowl|      761|
|     Chicken Burrito|      591|
| Chips and Guacamole|      506|
|       Steak Burrito|      386|
|   Canned Soft Drink|      351|
|               Chips|      230|
|          Steak Bowl|      221|
|       Bottled Water|      211|
|Chips and Fresh T...|      130|
|         Canned Soda|      126|
+--------------------+---------+
only showing top 10 rows



In [25]:
most_ordered.select("total_qty", "item_name").orderBy(col("total_qty").desc()).limit(1).show()

+---------+------------+
|total_qty|   item_name|
+---------+------------+
|      761|Chicken Bowl|
+---------+------------+



In [29]:
chipo.groupBy(col("choice_description")).agg(count("choice_description").alias("qty")).orderBy(col("qty").desc()).where("choice_description is not NULL").limit(1).show()

[Stage 46:>                                                         (0 + 1) / 1]

+------------------+----+
|choice_description| qty|
+------------------+----+
|              NULL|1246|
+------------------+----+



In [32]:
from pyspark.sql.functions import col, count

most_choice = (
    chipo
    .filter((col("choice_description").isNotNull()) & (col("choice_description") != "NULL"))
    .groupBy("choice_description")
    .agg(count("*").alias("qty"))
    .orderBy(col("qty").desc())
    .limit(1)
)

most_choice.show()

[Stage 52:>                                                         (0 + 1) / 1]

+------------------+---+
|choice_description|qty|
+------------------+---+
|       [Diet Coke]|134|
+------------------+---+

